# Phase 7: Prior Case Retrieval: GQA decoder

This is the **Prior Case Retrieval** part of the full `gqa` notebook (`notebooks/CS781_gqa_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Prior Case Retrieval

In [ ]:
import json, os
from huggingface_hub import snapshot_download
P7_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_7"; os.makedirs(P7_DIR, exist_ok=True)
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_7_PCR/*", "*/Phase_7_PCR/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_7" in r: return os.path.join(r, name)
files = {n: find(n + ".json") for n in ["train", "train_candidates", "dev", "dev_candidates", "test", "test_candidates"]}
if not any(files.values()): raise SystemExit("Phase 7 files are not in the repo yet")
def short(x, n=100):
    if isinstance(x, str): return x[:n] + (f"... [{len(x)} chars]" if len(x) > n else "")
    if isinstance(x, list): return f"list[{len(x)}] " + str([short(i, 50) for i in x[:2]])
    if isinstance(x, dict): return {k: short(v, 50) for k, v in list(x.items())[:4]}
    return x
D = {}
for k, p in files.items():
    if not p: print(f"\n== {k}: NOT FOUND"); continue
    D[k] = json.load(open(p)); d = D[k]
    print(f"\n== {k}: {os.path.getsize(p)/1e6:.1f} MB | {type(d).__name__} with {len(d)} items")
    ex = d[0] if isinstance(d, list) else next(iter(d.items()))
    print("   first item:", {kk: short(v) for kk, v in ex.items()} if isinstance(ex, dict) else short(ex))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


== train: 33.2 MB | dict with 827 items
   first item: ('0000000599', {'text': ['PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 1545 1964 SCR (3) 573 CITATOR INFO : R 1970 SC 765 (9) R 1971 SC 342 (6) RF 1974 SC1185 (16) F 1984 SC 305 (9,21) R 1991 SC1557 (20,22) ACT: Election Dispute-Joinder of -parties-Joinder of candi- date who did not contest-If invalidates eletion petition- "Copy", meaning of-Defects in verification and affidavit- Maintainability of petition- Representation of the People Act , 1951 , ss. 81 , 82 , 83 , 90 . HEADNOTE: The validity of the election of the appellant to <ORG> at the third general elections held in the month of February, 1962, was challenged by two of the electors of the constituency from which the appellant was elected, by filing election petitions for s

In [ ]:
import numpy as np, random
assert "D" in globals(), "re-run the Phase 7 inspection cell first"
print("split sizes:", {k: len(v) for k, v in D.items()})
for sp in ["train", "dev"]:
    q, c = D[sp], D[sp + "_candidates"]; pool = set(c)
    g = np.array([len(r["relevant_candidates"]) for r in q.values()]); allg = [x for r in q.values() for x in r["relevant_candidates"]]
    print(f"\n[{sp}] gold per query: mean {g.mean():.2f} | median {np.median(g):.0f} | min {g.min()} | p90 {np.percentile(g, 90):.0f} | max {g.max()}")
    print(f"  gold ids found in own pool: {100*np.mean([x in pool for x in allg]):.1f}% | pool docs gold for >=1 query: {100*len(set(allg) & pool)/len(pool):.1f}% | gold links per gold doc: {len(allg)/max(1, len(set(allg))):.2f}")
    print(f"  query ids that also appear in their own pool: {sum(k in pool for k in q)} of {len(q)}")
    hits = 7 * g / len(pool); P, R = hits.sum() / (7 * len(g)), hits.sum() / g.sum()
    print(f"  random-ranking Micro-F1@7 ≈ {2*P*R/(P+R):.4f} (P {P:.4f}, R {R:.4f})")
Pl = {sp: set(D[sp + "_candidates"]) for sp in ["train", "dev", "test"]}
print("\npool overlap train&dev:", len(Pl["train"] & Pl["dev"]), "| train&test:", len(Pl["train"] & Pl["test"]), "| dev&test:", len(Pl["dev"] & Pl["test"]))
print("query overlap train&dev:", len(set(D["train"]) & set(D["dev"])), "| train&test:", len(set(D["train"]) & set(D["test"])), "| dev&test:", len(set(D["dev"]) & set(D["test"])))
rng = random.Random(0); tot = 0
for name, d in [("train query", D["train"]), ("train cand", D["train_candidates"]), ("dev query", D["dev"]), ("dev cand", D["dev_candidates"]), ("test query", D["test"]), ("test cand", D["test_candidates"])]:
    recs = rng.sample(list(d.values()), min(150, len(d)))
    t = np.array([len(e.ids) for e in tokenizer.encode_batch([" ".join(r["text"])[:400000] for r in recs])]); tot += t.mean() * len(d)
    hdr = np.mean([r["text"][0].lstrip().startswith("PETITIONER") for r in d.values()])
    print(f"{name:12s} tokens p10/50/90/max: {int(np.percentile(t,10))}/{int(np.median(t))}/{int(np.percentile(t,90))}/{t.max()} | sentences (median) {int(np.median([len(r['text']) for r in recs]))} | start with 'PETITIONER': {100*hdr:.0f}%")
print(f"\nestimated total tokens across all six files: {tot/1e6:.0f} M")

split sizes: {'train': 827, 'train_candidates': 4320, 'dev': 118, 'dev_candidates': 1023, 'test': 237, 'test_candidates': 1727}

[train] gold per query: mean 6.42 | median 4 | min 0 | p90 14 | max 52
  gold ids found in own pool: 100.0% | pool docs gold for >=1 query: 95.1% | gold links per gold doc: 1.29
  query ids that also appear in their own pool: 827 of 827
  random-ranking Micro-F1@7 ≈ 0.0016 (P 0.0015, R 0.0016)

[dev] gold per query: mean 8.25 | median 5 | min 1 | p90 17 | max 92
  gold ids found in own pool: 100.0% | pool docs gold for >=1 query: 89.9% | gold links per gold doc: 1.06
  query ids that also appear in their own pool: 118 of 118
  random-ranking Micro-F1@7 ≈ 0.0074 (P 0.0081, R 0.0068)

pool overlap train&dev: 436 | train&test: 740 | dev&test: 220
query overlap train&dev: 0 | train&test: 0 | dev&test: 0
train query  tokens p10/50/90/max: 1899/5863/17328/61879 | sentences (median) 39 | start with 'PETITIONER': 34%
train cand   tokens p10/50/90/max: 1222/4100/13402

In [ ]:
import os, pickle, time, numpy as np
CACHE7 = f"{P7_DIR}/pcr_tok_cache.pkl"
if os.path.exists(CACHE7):
    C7 = pickle.load(open(CACHE7, "rb")); print("loaded cache from Drive")
else:
    assert "D" in globals(), "re-run the Phase 7 inspection cell first"
    t0 = time.time(); C7 = {}
    for name, d in D.items():
        keys = list(d.keys()); out = {}
        for i in range(0, len(keys), 32):
            batch = keys[i:i + 32]
            for k, e in zip(batch, tokenizer.encode_batch([" ".join(d[k]["text"]) for k in batch])):
                out[k] = np.array(e.ids, dtype=np.uint16)
        C7[name] = out; print(f"{name:17s} {len(out):5d} docs | {sum(len(v) for v in out.values())/1e6:5.1f} M tokens | {(time.time()-t0)/60:.1f} min")
    C7["gold"] = {sp: {k: r["relevant_candidates"] for k, r in D[sp].items()} for sp in ["train", "dev"]}
    pickle.dump(C7, open(CACHE7, "wb")); print("saved", CACHE7)
for sp in ["train", "dev", "test"]:
    pool, q = C7[sp + "_candidates"], C7[sp]
    msg = f"[{sp}] queries inside their own pool: {sum(k in pool for k in q)}/{len(q)}"
    if sp != "test": msg += f" | queries listing themselves as gold: {sum(k in v for k, v in C7['gold'][sp].items())}"
    print(msg)
print("train queries with zero gold:", sum(len(v) == 0 for v in C7["gold"]["train"].values()))

train               827 docs |   7.1 M tokens | 0.1 min
train_candidates   4320 docs |  34.2 M tokens | 0.7 min
dev                 118 docs |   1.2 M tokens | 0.7 min
dev_candidates     1023 docs |  11.0 M tokens | 1.0 min
test                237 docs |   2.0 M tokens | 1.0 min
test_candidates    1727 docs |  15.1 M tokens | 1.3 min
saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/pcr_tok_cache.pkl
[train] queries inside their own pool: 827/827 | queries listing themselves as gold: 0
[dev] queries inside their own pool: 118/118 | queries listing themselves as gold: 0
[test] queries inside their own pool: 237/237
train queries with zero gold: 10


In [ ]:
import time, numpy as np, torch, torch.nn.functional as F

@torch.no_grad()
def embed_split(bb, tok, ids=None, chunk=1024, bs=32):
    """Return {doc_id: (n_chunks, 2d) array}: one [last-token ; mean-token] vector per 1024-token chunk."""
    bb.eval(); ids = list(ids or tok.keys()); pieces = []
    for di, k in enumerate(ids):
        a = tok[k]
        for s in range(0, len(a), chunk): pieces.append((di, a[s:s + chunk]))
    order = sorted(range(len(pieces)), key=lambda i: len(pieces[i][1]))        # similar lengths together = little padding
    out = [None] * len(pieces)
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]; lens = [len(pieces[i][1]) for i in idx]; T = max(lens)
        x = torch.zeros(len(idx), T, dtype=torch.long)
        for j, i in enumerate(idx): x[j, :lens[j]] = torch.from_numpy(pieces[i][1].astype(np.int64))
        with torch.autocast("cuda", dtype=torch.bfloat16):
            f = bb.features(x.to(device), torch.tensor(lens, device=device))
        f = f.float().cpu().numpy()
        for j, i in enumerate(idx): out[i] = f[j]
    per_doc = {k: [] for k in ids}
    for (di, _), v in zip(pieces, out): per_doc[ids[di]].append(v)
    return {k: np.stack(v) for k, v in per_doc.items()}

def doc_matrix(ch, ids, part, pool, d):
    sl = slice(0, d) if part == "last" else slice(d, 2 * d)
    rows = []
    for k in ids:
        m = ch[k][:, sl]; rows.append(m.mean(0) if pool == "mean" else m.max(0) if pool == "max" else m[0])
    return np.stack(rows)

def f1_at_k(Qm, Cm, qids, cids, gold, center=False, k=7):
    Q, C = torch.tensor(Qm, device=device), torch.tensor(Cm, device=device)
    if center: mu = C.mean(0, keepdim=True); Q, C = Q - mu, C - mu
    S = F.normalize(Q, dim=-1) @ F.normalize(C, dim=-1).T
    pos = {c: i for i, c in enumerate(cids)}
    for qi, q in enumerate(qids):
        if q in pos: S[qi, pos[q]] = -1e9                      # a query must never retrieve itself
    top = S.topk(k, dim=1).indices.cpu().numpy()
    tp = sum(len({cids[j] for j in top[qi]} & set(gold[q])) for qi, q in enumerate(qids))
    P, R = tp / (k * len(qids)), tp / sum(len(gold[q]) for q in qids)
    return (2 * P * R / (P + R) if tp else 0.0), P, R

# ---- zero-shot baseline on dev ----
bb = make_backbone(); d = bb.d; t0 = time.time()
chq, chc = embed_split(bb, C7["dev"]), embed_split(bb, C7["dev_candidates"])
print(f"embedded dev queries + pool in {(time.time()-t0)/60:.1f} min")
qids, cids, gold = list(C7["dev"]), list(C7["dev_candidates"]), C7["gold"]["dev"]
print("random-ranking floor on dev is about 0.0074\n")
print(f"{'feature':8s} {'chunk pool':10s} {'centred':8s}   F1@7    P@7     R@7")
for part in ("mean", "last"):
    for pool in ("mean", "max", "first"):
        for center in (False, True):
            f, p, r = f1_at_k(doc_matrix(chq, qids, part, pool, d), doc_matrix(chc, cids, part, pool, d), qids, cids, gold, center)
            print(f"{part:8s} {pool:10s} {str(center):8s}  {f:.4f}  {p:.4f}  {r:.4f}")

embedded dev queries + pool in 1.0 min
random-ranking floor on dev is about 0.0074

feature  chunk pool centred    F1@7    P@7     R@7
mean     mean       False     0.2957  0.3220  0.2734
mean     mean       True      0.3068  0.3341  0.2837
mean     max        False     0.2123  0.2312  0.1963
mean     max        True      0.2312  0.2518  0.2138
mean     first      False     0.2135  0.2324  0.1973
mean     first      True      0.2101  0.2288  0.1942
last     mean       False     0.0867  0.0944  0.0802
last     mean       True      0.0867  0.0944  0.0802
last     max        False     0.0267  0.0291  0.0247
last     max        True      0.0278  0.0303  0.0257
last     first      False     0.0233  0.0254  0.0216
last     first      True      0.0200  0.0218  0.0185


In [ ]:
import pickle, os
HN_PATH = f"{P7_DIR}/hardneg_zs.pkl"
if os.path.exists(HN_PATH): HN = pickle.load(open(HN_PATH, "rb")); print("loaded hard negatives for", len(HN), "queries")
else:
    bb = make_backbone(); d = bb.d; t0 = time.time()
    chq, chc = embed_split(bb, C7["train"]), embed_split(bb, C7["train_candidates"])
    qids, cids, gold = list(C7["train"]), list(C7["train_candidates"]), C7["gold"]["train"]
    Qm, Cm = doc_matrix(chq, qids, "mean", "mean", d), doc_matrix(chc, cids, "mean", "mean", d)
    f, p, r = f1_at_k(Qm, Cm, qids, cids, gold, center=True)
    print(f"embedded train in {(time.time()-t0)/60:.1f} min | zero-shot train F1@7 {f:.4f}")
    Cc = torch.tensor(Cm, device=device); mu = Cc.mean(0, keepdim=True)
    S = F.normalize(torch.tensor(Qm, device=device) - mu, dim=-1) @ F.normalize(Cc - mu, dim=-1).T
    pos = {c: i for i, c in enumerate(cids)}; HN = {}
    for qi, q in enumerate(qids):
        if q in pos: S[qi, pos[q]] = -1e9
        for g in gold[q]: S[qi, pos[g]] = -1e9
        HN[q] = [cids[j] for j in S[qi].topk(50).indices.tolist()]
    pickle.dump(HN, open(HN_PATH, "wb")); print("saved", HN_PATH); del bb; torch.cuda.empty_cache()

embedded train in 3.3 min | zero-shot train F1@7 0.1846
saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/hardneg_zs.pkl


In [ ]:
import random, json, torch.nn as nn
R7 = globals().get("R7", {}); HN = pickle.load(open(f"{P7_DIR}/hardneg_zs.pkl", "rb"))
TRQ = [q for q, g in C7["gold"]["train"].items() if g]; print("training queries:", len(TRQ))

class Proj(nn.Module):                                # identity at init: starts exactly at the zero-shot embedding
    def __init__(self, d):
        super().__init__(); self.lin = nn.Linear(d, d); nn.init.eye_(self.lin.weight); nn.init.zeros_(self.lin.bias)
    def forward(self, x): return self.lin(x)

def rand_chunk(a, L, rng):
    if len(a) <= L: return a
    s = rng.randint(0, len(a) - L); return a[s:s + L]

def feats(bb, proj, docs, L, K, rng):                 # differentiable: K random L-token windows per doc, mean-pooled
    chunks = [rand_chunk(a, L, rng) for a in docs for _ in range(K)]
    lens = [len(c) for c in chunks]; x = torch.zeros(len(chunks), max(lens), dtype=torch.long)
    for j, c in enumerate(chunks): x[j, :lens[j]] = torch.from_numpy(c.astype(np.int64))
    with torch.autocast("cuda", dtype=torch.bfloat16):
        f = bb.features(x.to(device), torch.tensor(lens, device=device))[:, bb.d:]
    return F.normalize(proj(f.float().view(len(docs), K, -1).mean(1)), dim=-1)

@torch.no_grad()
def embed_final(bb, proj, tok):
    ch = embed_split(bb, tok); ids = list(tok.keys())
    return ids, proj(torch.tensor(doc_matrix(ch, ids, "mean", "mean", bb.d), device=device)).cpu().numpy()

def eval_dev(bb, proj):
    bb.eval(); qi, Qm = embed_final(bb, proj, C7["dev"]); ci, Cm = embed_final(bb, proj, C7["dev_candidates"])
    return f1_at_k(Qm, Cm, qi, ci, C7["gold"]["dev"], center=False)

def train_pcr(bb, name, lr, steps=600, B=8, L=512, K=1, tau=0.05, hn_top=30, lr_head=1e-3, eval_every=200, debug=False):
    proj = Proj(bb.d).to(device); rng = random.Random(0); params = [p for p in bb.parameters() if p.requires_grad]
    ntr = sum(p.numel() for p in params) + sum(p.numel() for p in proj.parameters())
    opt = torch.optim.AdamW([dict(params=list(proj.parameters()), lr=lr_head)] + ([dict(params=params, lr=lr)] if params else []), weight_decay=0.01)
    steps, eval_every = (4, 4) if debug else (steps, eval_every); warm = max(1, int(0.06 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    gold, pool, qtok = C7["gold"]["train"], C7["train_candidates"], C7["train"]
    best, t0, run, path = -1.0, time.time(), [], f"{P7_DIR}/pcr_{name}_best.pt"; bb.train(); torch.cuda.reset_peak_memory_stats()
    print(f"[{name}] trainable {ntr/1e6:.2f}M | steps {steps} | B {B} | window {L} x {K}")
    for step in range(1, steps + 1):
        qs = rng.sample(TRQ, B); pos_ids = [rng.choice(gold[q]) for q in qs]; hn_ids = [rng.choice(HN[q][:hn_top]) for q in qs]
        z = feats(bb, proj, [qtok[q] for q in qs] + [pool[p] for p in pos_ids] + [pool[h] for h in hn_ids], L, K, rng)
        logits = z[:B] @ z[B:].T / tau; cols = pos_ids + hn_ids
        mask = torch.zeros_like(logits, dtype=torch.bool)
        for i, q in enumerate(qs):
            gs = set(gold[q])
            for j, c in enumerate(cols):
                if j != i and c in gs: mask[i, j] = True          # also gold for this query: not a negative
        loss = F.cross_entropy(logits.masked_fill(mask, -1e4), torch.arange(B, device=device))
        loss.backward(); torch.nn.utils.clip_grad_norm_(params + list(proj.parameters()), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True); run.append(loss.item())
        if step % eval_every == 0 or step == steps:
            f, p, r = eval_dev(bb, proj); bb.train()
            print(f"step {step}/{steps} | loss {np.mean(run):.3f} | dev F1@7 {f:.4f} (P {p:.4f} R {r:.4f}) | {(time.time()-t0)/60:.1f} min"); run = []
            if f > best:
                best = f; torch.save(dict(bb={n: q.detach().cpu() for n, q in bb.named_parameters() if q.requires_grad}, proj=proj.state_dict()), path)
    ck = torch.load(path); bb.load_state_dict(ck["bb"], strict=False); proj.load_state_dict(ck["proj"])
    R7[name] = dict(dev_f1=best, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best dev F1@7 {best:.4f} | peak {R7[name]['peak_gb']:.1f} GB | {R7[name]['minutes']:.1f} min"); return bb, proj

@torch.no_grad()
def write_pred_pcr(bb, proj, name):
    bb.eval(); qi, Qm = embed_final(bb, proj, C7["test"]); ci, Cm = embed_final(bb, proj, C7["test_candidates"])
    S = F.normalize(torch.tensor(Qm, device=device), dim=-1) @ F.normalize(torch.tensor(Cm, device=device), dim=-1).T
    pos = {c: i for i, c in enumerate(ci)}
    for i, q in enumerate(qi):
        if q in pos: S[i, pos[q]] = -1e9                                  # never retrieve the query itself
    top = S.topk(7, dim=1).indices.cpu().numpy(); pred = {q: [ci[j] for j in top[i]] for i, q in enumerate(qi)}
    assert len(pred) == len(C7["test"]) and all(len(set(v)) == 7 and set(v) <= set(ci) and q not in v for q, v in pred.items())
    p = f"{P7_DIR}/prediction_pcr_{name}.json"; json.dump(pred, open(p, "w")); print("✅ wrote", p, "-> rename to prediction.json, zip, upload")
print("✅ retrieval toolkit ready")

training queries: 817
✅ retrieval toolkit ready


In [ ]:
bb = make_backbone(); bb, proj = train_pcr(bb, "gqa_full_long", lr=2e-5, steps=1500, B=16, eval_every=250)
write_pred_pcr(bb, proj, "gqa_full_long")

[gqa_full_long] trainable 88.10M | steps 1500 | B 16 | window 512 x 1
step 250/1500 | loss 2.157 | dev F1@7 0.3280 (P 0.3571 R 0.3032) | 1.9 min
step 500/1500 | loss 1.821 | dev F1@7 0.3291 (P 0.3584 R 0.3042) | 3.9 min
step 750/1500 | loss 1.612 | dev F1@7 0.3391 (P 0.3692 R 0.3135) | 5.9 min
step 1000/1500 | loss 1.472 | dev F1@7 0.3480 (P 0.3789 R 0.3217) | 7.8 min
step 1250/1500 | loss 1.451 | dev F1@7 0.3457 (P 0.3765 R 0.3196) | 9.8 min
step 1500/1500 | loss 1.384 | dev F1@7 0.3457 (P 0.3765 R 0.3196) | 11.7 min

✅ [gqa_full_long] best dev F1@7 0.3480 | peak 22.0 GB | 11.7 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/prediction_pcr_gqa_full_long.json -> rename to prediction.json, zip, upload


In [ ]:
bb = make_backbone(); bb, proj = train_pcr(bb, "gqa_full", lr=2e-5)
write_pred_pcr(bb, proj, "gqa_full")

[gqa_full] trainable 88.10M | steps 600 | B 8 | window 512 x 1
step 200/600 | loss 1.872 | dev F1@7 0.3013 (P 0.3281 R 0.2785) | 1.4 min
step 400/600 | loss 1.592 | dev F1@7 0.3146 (P 0.3426 R 0.2909) | 2.8 min
step 600/600 | loss 1.445 | dev F1@7 0.3224 (P 0.3511 R 0.2980) | 4.2 min

✅ [gqa_full] best dev F1@7 0.3224 | peak 11.8 GB | 4.3 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/prediction_pcr_gqa_full.json -> rename to prediction.json, zip, upload


In [ ]:
def enc_chunks(bb, proj, chunks):                      # differentiable: list of token arrays -> (n, d) unit vectors
    lens = [len(c) for c in chunks]; x = torch.zeros(len(chunks), max(lens), dtype=torch.long)
    for j, c in enumerate(chunks): x[j, :lens[j]] = torch.from_numpy(c.astype(np.int64))
    with torch.autocast("cuda", dtype=torch.bfloat16):
        f = bb.features(x.to(device), torch.tensor(lens, device=device))[:, bb.d:]
    return F.normalize(proj(f.float()), dim=-1)

def load_pcr(name, kind="gqa"):
    m = make_backbone(kind); ck = torch.load(f"{P7_DIR}/pcr_{name}_best.pt")
    m.load_state_dict(ck["bb"], strict=False); p = Proj(m.d).to(device); p.load_state_dict(ck["proj"]); m.eval()
    for q in list(m.parameters()) + list(p.parameters()): q.requires_grad_(False)
    return m, p

def train_kd_pcr(student, teacher, tproj, name, lr=5e-5, steps=1500, B=16, L=512, tau=0.05, alpha=0.5, hn_top=30,
                 lr_head=1e-3, eval_every=250, debug=False):
    sproj = Proj(student.d).to(device); rng = random.Random(0); params = [p for p in student.parameters() if p.requires_grad]
    ntr = sum(p.numel() for p in params) + sum(p.numel() for p in sproj.parameters())
    t1 = time.time(); ft, _, _ = eval_dev(teacher, tproj); t_inf = time.time() - t1
    print(f"teacher dev F1@7 {ft:.4f} | teacher dev embedding time {t_inf:.0f}s | student trainable {ntr/1e6:.1f}M")
    opt = torch.optim.AdamW([dict(params=list(sproj.parameters()), lr=lr_head), dict(params=params, lr=lr)], weight_decay=0.01)
    steps, eval_every = (4, 4) if debug else (steps, eval_every); warm = max(1, int(0.06 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    gold, pool, qtok = C7["gold"]["train"], C7["train_candidates"], C7["train"]
    best, t0, run, path = -1.0, time.time(), [], f"{P7_DIR}/pcr_{name}_best.pt"; student.train(); torch.cuda.reset_peak_memory_stats()
    for step in range(1, steps + 1):
        qs = rng.sample(TRQ, B); pos_ids = [rng.choice(gold[q]) for q in qs]; hn_ids = [rng.choice(HN[q][:hn_top]) for q in qs]
        docs = [qtok[q] for q in qs] + [pool[p] for p in pos_ids] + [pool[h] for h in hn_ids]
        chunks = [rand_chunk(a, L, rng) for a in docs]                       # the same windows go to teacher and student
        zs = enc_chunks(student, sproj, chunks)
        with torch.no_grad(): zt = enc_chunks(teacher, tproj, chunks)
        ls, lt = zs[:B] @ zs[B:].T / tau, zt[:B] @ zt[B:].T / tau; cols = pos_ids + hn_ids
        mask = torch.zeros_like(ls, dtype=torch.bool)
        for i, q in enumerate(qs):
            gs = set(gold[q])
            for j, c in enumerate(cols):
                if j != i and c in gs: mask[i, j] = True
        ls, lt = ls.masked_fill(mask, -1e4), lt.masked_fill(mask, -1e4)
        ce = F.cross_entropy(ls, torch.arange(B, device=device))
        kd = F.kl_div(F.log_softmax(ls, -1), F.softmax(lt, -1), reduction="batchmean")
        loss = (1 - alpha) * ce + alpha * kd
        loss.backward(); torch.nn.utils.clip_grad_norm_(params + list(sproj.parameters()), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True); run.append((ce.item(), kd.item()))
        if step % eval_every == 0 or step == steps:
            f, p, r = eval_dev(student, sproj); student.train(); a = np.mean(run, 0)
            print(f"step {step}/{steps} | CE {a[0]:.3f} | KD {a[1]:.3f} | dev F1@7 {f:.4f} (P {p:.4f} R {r:.4f}) | {(time.time()-t0)/60:.1f} min"); run = []
            if f > best:
                best = f; torch.save(dict(bb={n: q.detach().cpu() for n, q in student.named_parameters() if q.requires_grad}, proj=sproj.state_dict()), path)
    ck = torch.load(path); student.load_state_dict(ck["bb"], strict=False); sproj.load_state_dict(ck["proj"])
    t1 = time.time(); eval_dev(student, sproj); s_inf = time.time() - t1
    R7[name] = dict(dev_f1=best, teacher_f1=ft, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60, student_embed_s=s_inf, teacher_embed_s=t_inf)
    print(f"\n✅ [{name}] student dev F1@7 {best:.4f} vs teacher {ft:.4f} | dev embedding time: student {s_inf:.0f}s vs teacher {t_inf:.0f}s")
    return student, sproj
print("✅ KD toolkit ready")

✅ KD toolkit ready


In [ ]:
teacher, tproj = load_pcr("gqa_full"); st = make_backbone("student")
train_kd_pcr(st, teacher, tproj, "dbg_kd", debug=False); del st; torch.cuda.empty_cache()

teacher dev F1@7 0.3224 | teacher dev embedding time 60s | student trainable 43.5M
step 4/4 | CE 2.453 | KD 0.663 | dev F1@7 0.2991 (P 0.3257 R 0.2765) | 0.7 min

✅ [dbg_kd] student dev F1@7 0.2991 vs teacher 0.3224 | dev embedding time: student 40s vs teacher 60s


In [ ]:
st = make_backbone("student"); st, sproj = train_kd_pcr(st, teacher, tproj, "student_kd")
write_pred_pcr(st, sproj, "student_kd")

In [ ]:
bb = make_backbone("student"); bb, p = train_pcr(bb, "student_ce", lr=5e-5, steps=1500, B=16, eval_every=250); del bb; torch.cuda.empty_cache()